In [1]:
import os

In [13]:
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
from docling.datamodel.pipeline_options import PdfPipelineOptions

In [4]:
hr_ploicy_file=r"C:\dev\lv-assignment1\zdata\ABC_Corporation_HR_Policy_Manual.pdf"
law_policy_file=r"C:\dev\lv-assignment1\zdata\Historical_Court_Decisions_Compilation.pdf"

In [14]:
pdf_option = PdfPipelineOptions(
    do_ocr=False,
    do_table_structure=True
)

In [15]:
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_option)}
)

In [16]:
hr_policy_doc=converter.convert(hr_ploicy_file)

In [19]:
hr_policy_doc.document

DoclingDocument(schema_name='DoclingDocument', version='1.9.0', name='ABC_Corporation_HR_Policy_Manual', origin=DocumentOrigin(mimetype='application/pdf', binary_hash=4658216403192749416, filename='ABC_Corporation_HR_Policy_Manual.pdf', uri=None), furniture=GroupItem(self_ref='#/furniture', parent=None, children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>), body=GroupItem(self_ref='#/body', parent=None, children=[RefItem(cref='#/texts/0'), RefItem(cref='#/texts/1'), RefItem(cref='#/texts/2'), RefItem(cref='#/texts/3'), RefItem(cref='#/texts/4'), RefItem(cref='#/texts/5'), RefItem(cref='#/texts/6'), RefItem(cref='#/texts/7'), RefItem(cref='#/texts/8'), RefItem(cref='#/texts/9'), RefItem(cref='#/texts/10'), RefItem(cref='#/texts/11'), RefItem(cref='#/texts/12'), RefItem(cref='#/texts/13'), RefItem(cref='#/texts/14'), RefItem(cref='#/texts/15'), RefItem(cref='#/texts/16'), RefItem(cref='#/texts/17'), RefI

In [22]:
len(hr_policy_doc.pages)

2

In [23]:
print(hr_policy_doc.document.export_to_markdown())

## ABC Corporation

## Human Resources Policy Manual (2026 Edition)

## 1. Introduction

ABC Corporation is committed to fostering a professional, inclusive, and high n performance workplace. This HR Policy Manual outlines the principles, procedures, and standards governing employment practices. These policies apply to all employees, contractors, and interns working with the organization.

## 2. Equal Employment Opportunity (EEO)

ABC Corporation provides equal employment opportunities to all individuals without regard to race, gender, religion, nationality, disability, age, or any other protected category. Discrimination, harassment, and retaliation are strictly prohibited and may result in disciplinary action.

## 3. Recruitment and Selection

All hiring decisions are based on merit, qualifications, and business needs. The recruitment process includes job posting, screening, interviews, background verification, and reference checks. HR ensures transparency and fairness at every stage

In [29]:
import tiktoken
tiktoken_encoder = tiktoken.get_encoding("cl100k_base")
tokenizer = OpenAITokenizer(tokenizer=tiktoken_encoder,max_tokens=512)

In [30]:
chunker = HybridChunker(tokenizer=tokenizer, chunk_size=512, chunk_overlap=50,merge_peers=True)

In [33]:
raw_chunks = list(chunker.chunk(hr_policy_doc.document))

In [34]:
raw_chunks

[DocChunk(text='ABC Corporation is committed to fostering a professional, inclusive, and high n performance workplace. This HR Policy Manual outlines the principles, procedures, and standards governing employment practices. These policies apply to all employees, contractors, and interns working with the organization.', meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[DocItem(self_ref='#/texts/3', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=78.0, t=609.2899705078124, r=568.0, b=563.0199705078124, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 302))], comments=[])], headings=['1. Introduction'], captions=None, origin=DocumentOrigin(mimetype='application/pdf', binary_hash=4658216403192749416, filename='ABC_Corporation_HR_Policy_Manual.pdf', uri=None))),
 DocChunk(text='ABC Corporation 

In [61]:
result = []

for idx,chunk in enumerate(raw_chunks):
    headings=[]
    if chunk.meta.headings:
        headings=chunk.meta.headings
    page_no =[]
    if chunk.meta.doc_items:
        for item in chunk.meta.doc_items:
            for prov in item.prov:
                page_no.append(prov.page_no)
    captions=[]
    if chunk.meta.captions:
        captions=chunk.meta.captions

    result.append(
        {
            "filename":chunk.meta.origin.filename,
            "headings":headings,
            "page_no":page_no,
            "captions":captions,
            "text":chunk.text
        }
    )


C:\Users\ACER\AppData\Local\Temp\ipykernel_30592\2295604332.py:13: DeprecationWarning: deprecated
  if chunk.meta.captions:


In [62]:
result

[{'filename': 'ABC_Corporation_HR_Policy_Manual.pdf',
  'headings': ['1. Introduction'],
  'page_no': [1],
  'captions': [],
  'text': 'ABC Corporation is committed to fostering a professional, inclusive, and high n performance workplace. This HR Policy Manual outlines the principles, procedures, and standards governing employment practices. These policies apply to all employees, contractors, and interns working with the organization.'},
 {'filename': 'ABC_Corporation_HR_Policy_Manual.pdf',
  'headings': ['2. Equal Employment Opportunity (EEO)'],
  'page_no': [1],
  'captions': [],
  'text': 'ABC Corporation provides equal employment opportunities to all individuals without regard to race, gender, religion, nationality, disability, age, or any other protected category. Discrimination, harassment, and retaliation are strictly prohibited and may result in disciplinary action.'},
 {'filename': 'ABC_Corporation_HR_Policy_Manual.pdf',
  'headings': ['3. Recruitment and Selection'],
  'page_

In [67]:
embedding_model = "text-embedding-3-small"
dimension=1536

In [92]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

In [75]:
embedding_client = OpenAI(api_key=openai_api_key)

In [78]:
embedding =[]
for chunk in result:
    response = embedding_client.embeddings.create(
        input=chunk["text"],
        model=embedding_model,
        encoding_format="float"
    )
    embedding.append(response.data[0].embedding)

In [93]:
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_environment = os.getenv("PINECONE_ENVIRONMENT")
pinecone_index_name = os.getenv("PINECONE_INDEX_NAME")


In [94]:
pinecone_index_name

'hr-policy'

In [95]:
from pinecone.grpc import PineconeGRPC

In [96]:
pc=PineconeGRPC(api_key=pinecone_api_key)

In [103]:
index_description = pc.describe_index(name=pinecone_index_name)

In [104]:
index_description.host

'hr-policy-rev8dya.svc.aped-4627-b74a.pinecone.io'

In [105]:
index = pc.Index(host=index_description.host)

In [132]:
index.delete(delete_all=True,namespace="")

{'_response_info': {'raw_headers': {'date': 'Thu, 19 Feb 2026 16:09:18 GMT',
   'x-pinecone-request-latency-ms': '127',
   'x-envoy-upstream-service-time': '128',
   'x-pinecone-response-duration-ms': '129',
   'server': 'envoy'}}}

In [133]:
for chunk, embed in zip(result, embedding):
    index.upsert(
        vectors=[
            {
                "id": f"{chunk['filename']}_{chunk['page_no']}_{chunk['headings']}",
                "values": embed,
                "metadata": {
                    "filename": chunk["filename"],
                    "headings": chunk["headings"],
                    "page_no": int(chunk["page_no"][0]) if isinstance(chunk["page_no"], list) else int(chunk["page_no"]),
                    "captions": chunk["captions"],
                    "text": chunk["text"]
                }
            }
        ]
    )

In [134]:
question="what is the working hours mentioned in the document?"

In [135]:
question

'what is the working hours mentioned in the document?'

In [152]:
def HR_retrieval(question):
    query_emded=embedding_client.embeddings.create(
        input=question,
        model=embedding_model,
        encoding_format="float"
    )
    query_vector=query_emded.data[0].embedding
    answer = index.query(
        vector=query_vector,
        top_k=2,
        include_metadata=True
    )
    return answer.matches[0].metadata["text"]

In [153]:
HR_retrieval("when is the performance review done?")

'Performance reviews are conducted bi n annually. Employees receive structured feedback, goal alignment, and development planning. Outstanding performers may receive incentives and career advancement opportunities.'